In [ ]:
import sys
sys.path.append('/data/lhz/code/OSSL-Classification')
import models.clip.clip as clip
model,ppp = clip.load("ViT-B/16", device="cuda")
cifar100_mean, cifar100_std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761) 


from torchvision import transforms
test_transforms = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar100_mean, std=cifar100_std)
])  

from torchvision.datasets import CIFAR100
import torch
datasets = CIFAR100(root='/home/lhz/data', train=False, download=False, transform=test_transforms)
dataset_loaders = torch.utils.data.DataLoader(datasets, batch_size=128, shuffle=False, num_workers=0)


class_names = datasets.classes
tokenizer = clip.tokenize

In [16]:
model = model.float().eval()
model = model.cuda()

In [17]:
with torch.no_grad():
    all_text = [f"a photo of a {c}" for c in class_names]
    text = tokenizer(all_text).cuda()
    text_features = model.encode_text(text)

In [18]:
all_labels = []
all_preds = []
with torch.no_grad():
    for image,label in dataset_loaders:
        image = image.cuda()
        image_embeds = model.encode_image(image)
        image_embeds /= image_embeds.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        logits = image_embeds @ text_features.T
        preds = logits.argmax(dim=-1)
        all_labels.append(label)
        all_preds.append(preds)
        

In [19]:
all_labels = torch.cat(all_labels)
all_preds = torch.cat(all_preds)
all_preds = all_preds.cpu()
(all_labels==all_preds).sum()

tensor(6460)